In [1]:
!pip install opencv-python

In [2]:
import cv2
import numpy as np

import matplotlib.pyplot as plt
from PIL import Image

# Ocean Wave Energy Mapper

This project analyzes a video of ocean waves to visualize accumulated
motion intensity over time, highlighting the most turbulent zone —
a technique with real applications in coastal monitoring for identifying
potentially hazardous areas (e.g., rip currents) along a beach.

## Setup

Loading the video and initializing the accumulator matrix (used to track
motion intensity per pixel) and the video writer for the final output.

In [3]:
capture = cv2.VideoCapture("ocean_waves.mp4")
ret, frame = capture.read()
prev_gray = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)

In [4]:
frame_height, frame_width = prev_gray.shape
matrix = np.zeros(shape=(frame_height, frame_width), dtype=np.float32)

fps = capture.get(cv2.CAP_PROP_FPS)

In [5]:
fourcc = cv2.VideoWriter_fourcc(*'H264')
out = cv2.VideoWriter('/content/wave_energy_map.mp4', fourcc, fps, (frame_width, frame_height))

## Main Processing Loop

For each frame:
1. Compute the absolute difference from the previous frame (frame differencing)
2. Threshold the difference to isolate meaningful motion (filtering out noise)
3. Accumulate motion intensity per pixel over time
4. Normalize and apply a color map to visualize motion as a heatmap
5. Blend the heatmap with the original frame
6. Detect and highlight the region with the highest accumulated motion

In [6]:
while True:
    ret, frame = capture.read()
    if not ret:
        break

    current_gray = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)
    difference = cv2.absdiff(current_gray, prev_gray)

    _, binary = cv2.threshold(difference, 20, 255, cv2.THRESH_BINARY)
    matrix += binary / 255

    prev_gray = current_gray

    display_matrix = np.clip(matrix, 0, 60)
    normalized = cv2.normalize(display_matrix, None, 0, 255, cv2.NORM_MINMAX)
    normalized = normalized.astype(np.uint8)

    heatmap = cv2.applyColorMap(normalized, cv2.COLORMAP_OCEAN)
    combined = cv2.addWeighted(heatmap, 0.5, frame, 0.5, 0)

    _, high_motion = cv2.threshold(normalized, 130, 255, cv2.THRESH_BINARY)
    contours, _ = cv2.findContours(high_motion, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)

    if contours:
        max_contour = max(contours, key=cv2.contourArea)
        x, y, w, h = cv2.boundingRect(max_contour)
        cv2.rectangle(combined, (x, y), (x+w, y+h), (0, 0, 255), 2)
        cv2.putText(combined, "High Wave Activity", (x, y-10),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.6, (0, 0, 255), 2)

    out.write(combined)

## Result

The final video (`wave_energy_map.mp4`) highlights the zone of highest
wave activity throughout the clip, useful as a lightweight visual indicator
of turbulence concentration along the shoreline.

In [7]:
capture.release()
out.release()
print("  Saved at /content/wave_energy_map.acv1")

  Saved at /content/wave_energy_map.acv1


## Sample Output

In [9]:
cap = cv2.VideoCapture("/content/wave_energy_map.mp4")
frames = []
max_frames = 100

count = 0
while count < max_frames:
    ret, frame = cap.read()
    if not ret:
        break
    frame_rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
    frames.append(Image.fromarray(frame_rgb))
    count += 1

cap.release()

frames[0].save("/content/sample_output.gif", save_all=True, append_images=frames[1:], duration=40, loop=0)

## Limitations
- Motion intensity reflects visual change, not actual wave height or
  water depth, so it's an approximation rather than a physical measurement.
- Best suited for fixed-camera footage; camera movement would introduce
  false motion readings.